# CC12M Image Downloader (Resume-Safe)

This notebook downloads all images listed in `dataset/cc12m.tsv` into `dataset/cc12m_image_cache`.

Behavior:
- Skips images that are already cached
- Retries failed downloads
- Logs failures to `dataset/cc12m_image_cache/_failed_downloads.tsv`
- Safe to rerun multiple times

In [ ]:
import os
import io
import time
import hashlib
from urllib.parse import urlparse
from urllib.request import Request, urlopen
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

from PIL import Image
from tqdm.auto import tqdm

TSV_PATH = 'dataset/cc12m.tsv'
CACHE_DIR = 'dataset/cc12m_image_cache'
NUM_WORKERS = min(64, (os.cpu_count() or 8) * 4)
MAX_PENDING = NUM_WORKERS * 4
TIMEOUT_SECONDS = 30
RETRIES = 3
VERIFY_EXISTING = False

print('TSV_PATH:', TSV_PATH)
print('CACHE_DIR:', CACHE_DIR)
print('NUM_WORKERS:', NUM_WORKERS)
print('MAX_PENDING:', MAX_PENDING)
print('TIMEOUT_SECONDS:', TIMEOUT_SECONDS)
print('RETRIES:', RETRIES)
print('VERIFY_EXISTING:', VERIFY_EXISTING)

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif'}


def cache_path_from_url(image_url, cache_dir):
    parsed = urlparse(str(image_url))
    ext = os.path.splitext(parsed.path)[1].lower()
    if ext not in VALID_EXTS:
        ext = '.img'
    key = hashlib.sha1(str(image_url).encode('utf-8')).hexdigest()
    return os.path.join(cache_dir, key + ext)


def is_cached(path, verify=False):
    if not os.path.isfile(path):
        return False
    if not bool(verify):
        return True
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        try:
            os.remove(path)
        except Exception:
            pass
        return False


def download_to_cache(image_url, cache_path, timeout_seconds=30, retries=3):
    if os.path.isfile(cache_path):
        return ('skipped', '')

    req = Request(str(image_url), headers={'User-Agent': 'Mozilla/5.0'})
    last_error = ''

    for attempt in range(max(1, int(retries))):
        tmp_path = cache_path + '.tmp'
        try:
            with urlopen(req, timeout=float(timeout_seconds)) as resp:
                data = resp.read()

            if not data:
                raise RuntimeError('empty response body')

            with Image.open(io.BytesIO(data)) as img:
                img.verify()

            with open(tmp_path, 'wb') as f:
                f.write(data)
            os.replace(tmp_path, cache_path)
            return ('downloaded', '')
        except Exception as exc:
            last_error = f'{type(exc).__name__}: {exc}'
            try:
                if os.path.exists(tmp_path):
                    os.remove(tmp_path)
            except Exception:
                pass
            if attempt + 1 < max(1, int(retries)):
                time.sleep(min(2 ** attempt, 5))

    return ('failed', last_error)


def run_cc12m_cache_download(
    tsv_path,
    cache_dir,
    num_workers=32,
    max_pending=128,
    timeout_seconds=30,
    retries=3,
    verify_existing=False,
):
    if not os.path.isfile(tsv_path):
        raise FileNotFoundError(f'CC12M TSV not found: {tsv_path}')

    os.makedirs(cache_dir, exist_ok=True)
    failed_log = os.path.join(cache_dir, '_failed_downloads.tsv')

    stats = {
        'total_rows': 0,
        'invalid_rows': 0,
        'skipped_existing': 0,
        'downloaded': 0,
        'failed': 0,
    }

    with open(tsv_path, 'r', encoding='utf-8') as src, \
         open(failed_log, 'a', encoding='utf-8') as failed_fp, \
         ThreadPoolExecutor(max_workers=int(num_workers)) as executor, \
         tqdm(desc='CC12M rows', unit='row') as pbar:

        futures = {}

        def handle_done(done_set):
            for fut in done_set:
                line_no, image_url, _ = futures.pop(fut)
                try:
                    status, err = fut.result()
                except Exception as exc:
                    status, err = ('failed', f'{type(exc).__name__}: {exc}')

                if status == 'downloaded':
                    stats['downloaded'] += 1
                elif status == 'skipped':
                    stats['skipped_existing'] += 1
                else:
                    stats['failed'] += 1
                    failed_fp.write(f'{line_no}\t{image_url}\t{err}\n')

                pbar.update(1)

        for line_no, raw_line in enumerate(src, start=1):
            stats['total_rows'] += 1
            line = raw_line.rstrip('\n')

            if (not line) or ('	' not in line):
                stats['invalid_rows'] += 1
                pbar.update(1)
                continue

            image_url, caption = line.split('	', 1)
            image_url = image_url.strip()
            caption = caption.strip()

            if (not image_url) or (not caption):
                stats['invalid_rows'] += 1
                pbar.update(1)
                continue

            cache_path = cache_path_from_url(image_url, cache_dir)
            if is_cached(cache_path, verify=verify_existing):
                stats['skipped_existing'] += 1
                pbar.update(1)
                continue

            fut = executor.submit(
                download_to_cache,
                image_url,
                cache_path,
                timeout_seconds,
                retries,
            )
            futures[fut] = (line_no, image_url, cache_path)

            if len(futures) >= int(max_pending):
                done_set, _ = wait(futures.keys(), return_when=FIRST_COMPLETED)
                handle_done(done_set)

        while futures:
            done_set, _ = wait(futures.keys(), return_when=FIRST_COMPLETED)
            handle_done(done_set)

    return stats, failed_log

In [ ]:
stats, failed_log_path = run_cc12m_cache_download(
    tsv_path=TSV_PATH,
    cache_dir=CACHE_DIR,
    num_workers=NUM_WORKERS,
    max_pending=MAX_PENDING,
    timeout_seconds=TIMEOUT_SECONDS,
    retries=RETRIES,
    verify_existing=VERIFY_EXISTING,
)

print('\nDownload summary')
print('total_rows       :', stats['total_rows'])
print('invalid_rows     :', stats['invalid_rows'])
print('skipped_existing :', stats['skipped_existing'])
print('downloaded       :', stats['downloaded'])
print('failed           :', stats['failed'])
if stats['failed'] > 0:
    print('Failed download log:', failed_log_path)